# PAP Emulation - PyBaMM Battery Models

Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary. Licensed under GNU GPL v3.0. Third-party dependencies retain their own licenses.

SPM, SPMe and DFN electrochemical battery model execution, parameter-set provenance, solver diagnostics and optional JAX device discovery.

**Author:** Abhishek Choudhary · **Affiliation:** AyeAI · **ORCID:** [0009-0002-0684-8320](https://orcid.org/0009-0002-0684-8320)

**Deployment:** browser-only platform; this notebook runs in a user-controlled Colab runtime. GPU/TPU availability is variable and not guaranteed. **Evidence boundary:** results remain `SIMULATED` and `UNVERIFIED`; this is not HIL or physical verification.

## 0. Runtime, security and provenance
Run cells in order. The notebook records backend versions, solver settings, runtime device, model configuration, timestamps and SHA-256 job/result digests. Results must be imported through the separate PAP Colab Bridge. Never upload untrusted native FMUs or binaries.

In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.
# AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
# Embedded in each notebook so notebooks remain independently runnable.
PAP_BRIDGE_VERSION = '1.0.0'
PAP_JOB_SCHEMA = 'pap.emulation.job/1.0'
PAP_RESULT_SCHEMA = 'pap.emulation.result/1.0'

def pap_canonical(value):
    import json
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False)

def pap_digest(text):
    import hashlib
    return 'sha256:' + hashlib.sha256(text.encode('utf-8')).hexdigest()

def pap_make_job(backend_id, operation, inputs):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_JOB_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'createdAt': datetime.now(timezone.utc).isoformat(),
        'target': {'backendId': backend_id, 'backendVersion': None, 'operation': operation, 'requestedDevice': 'auto'},
        'sir': {'header': {'id': 'colab-reference-fixture', 'irVersion': '1.0.0'}, 'components': []},
        'inputs': inputs,
        'observations': None,
        'configuration': {},
        'provenance': {'source': 'COLAB_REFERENCE_FIXTURE', 'sourceVersion': '1.0.0', 'license': 'GPL-3.0-only'}
    }
    canonical = pap_canonical(body)
    digest = pap_digest(canonical)
    return {**body, 'jobId': 'papjob-' + digest[7:27], 'digest': digest, 'canonicalPayload': canonical}

def pap_verify_job(job, expected_backend):
    import json
    if not isinstance(job, dict) or job.get('schema') != PAP_JOB_SCHEMA:
        raise ValueError('Unsupported/missing PAP job schema')
    payload = job.get('canonicalPayload')
    if not isinstance(payload, str): raise ValueError('Job canonicalPayload missing')
    parsed = json.loads(payload)
    body = {k:v for k,v in job.items() if k not in ('jobId','digest','canonicalPayload')}
    if pap_canonical(parsed) != pap_canonical(body): raise ValueError('Job payload/body mismatch')
    digest = pap_digest(payload)
    if digest != job.get('digest') or job.get('jobId') != 'papjob-' + digest[7:27]:
        raise ValueError('Job digest/jobId verification failed')
    if job.get('target',{}).get('backendId') != expected_backend:
        raise ValueError('Job targets a different backend')
    if not isinstance(job.get('inputs'), dict): raise ValueError('Job inputs must be an object')
    return job

def pap_get_job(backend_id, operation, default_inputs, use_demo=True):
    if use_demo:
        return pap_make_job(backend_id, operation, default_inputs), True
    from google.colab import files
    import json
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith('.json')]
    if len(candidates) != 1: raise ValueError('Upload exactly one browser-exported PAP job .json')
    return pap_verify_job(json.loads(uploaded[candidates[0]].decode('utf-8')), backend_id), False

def pap_make_result(job, backend_id, backend_version, outputs, diagnostics, device='cpu'):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_RESULT_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'jobId': job['jobId'],
        'jobDigest': job['digest'],
        'backend': {'id': backend_id, 'version': str(backend_version)},
        'execution': {'status': 'EXECUTED', 'runtime': 'COLAB', 'startedAt': None,
                      'finishedAt': datetime.now(timezone.utc).isoformat(), 'device': str(device)},
        'outputs': outputs,
        'diagnostics': diagnostics,
        'fidelity': 'SIMULATED',
        'physicalVerification': False,
        'evidenceState': 'UNVERIFIED',
        'claims': {'verified': False, 'hil': False, 'physical': False}
    }
    canonical = pap_canonical(body)
    return {**body, 'canonicalPayload': canonical, 'resultDigest': pap_digest(canonical)}

def pap_export(job, result, is_demo, result_filename='pap-emulation-result.json'):
    import json
    with open(result_filename, 'w', encoding='utf-8') as stream:
        json.dump(result, stream, indent=2, ensure_ascii=False, allow_nan=False)
    if is_demo:
        with open('pap-emulation-job.json', 'w', encoding='utf-8') as stream:
            json.dump(job, stream, indent=2, ensure_ascii=False, allow_nan=False)
    try:
        from google.colab import files
        files.download(result_filename)
        if is_demo: files.download('pap-emulation-job.json')
    except ImportError:
        print('Saved locally:', result_filename, '(download manually outside Colab)')
    print('Result remains SIMULATED / UNVERIFIED. Import it through the PAP Colab Bridge.')


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
%pip -q install 'pybamm>=24.11' 'numpy>=1.26,<3'

In [ ]:
import json, hashlib, platform, sys, numpy as np, pybamm
print({'python':sys.version,'platform':platform.platform(),'pybamm':pybamm.__version__})
PAP_USE_DEMO_JOB=True
default_inputs={'modelName':'SPM','parameterSet':'Chen2020','stopTimeSeconds':1800.0,'sampleCount':181}
job,job_is_demo=pap_get_job('pybamm.battery','pybamm.simulate',default_inputs,PAP_USE_DEMO_JOB)

## 1. Select electrochemical model fidelity
SPM is reduced-order; SPMe adds electrolyte dynamics; DFN is the Doyle-Fuller-Newman pseudo-two-dimensional model. They have different assumptions and computational cost. Cross-model agreement is not physical validation.

In [ ]:
name=job['inputs'].get('modelName','SPM'); model_ctor={'SPM':pybamm.lithium_ion.SPM,'SPMe':pybamm.lithium_ion.SPMe,'DFN':pybamm.lithium_ion.DFN}.get(name)
if model_ctor is None: raise ValueError('modelName must be SPM, SPMe or DFN')
model=model_ctor(); parameter_set=job['inputs'].get('parameterSet','Chen2020'); params=pybamm.ParameterValues(parameter_set)
solver=pybamm.CasadiSolver(mode='safe'); sim=pybamm.Simulation(model,parameter_values=params,solver=solver)
stop=float(job['inputs'].get('stopTimeSeconds',1800)); count=int(job['inputs'].get('sampleCount',181))
if stop<=0 or count<2 or count>100001: raise ValueError('Invalid stopTimeSeconds/sampleCount')
solution=sim.solve(t_eval=np.linspace(0,stop,count)); print('termination=',solution.termination,'samples=',len(solution.t))

## 2. Accelerator probe and result export
The PyBaMM simulation above uses CasADi and is labelled CPU. JAX device discovery is informative only; it does not mean this solve used a GPU. A separate JAX-compatible solver and model path must be explicitly configured and validated before claiming GPU execution.

In [ ]:
gpu={'jaxInstalled':False,'devices':[],'usedByThisSolve':False}
try:
 import jax; gpu={'jaxInstalled':True,'devices':[str(d) for d in jax.devices()],'backend':jax.default_backend(),'usedByThisSolve':False}
except Exception as e: gpu['error']=str(e)
vars=['Terminal voltage [V]','Current [A]','X-averaged cell temperature [K]']; obs={}
for var in vars:
 try:
  arr=np.asarray(solution[var].entries).reshape(-1)
  if np.isfinite(arr).all(): obs[var]=arr.tolist()
 except KeyError: pass
if not obs: raise RuntimeError('No expected observables found; inspect the selected model variables')
outputs=[{'sequence':int(i),'time':float(t),'values':{k:float(v[i]) for k,v in obs.items()}} for i,t in enumerate(solution.t)]
result=pap_make_result(job,'pybamm.battery',pybamm.__version__,outputs,{'model':name,'parameterSet':parameter_set,'solver':'CasadiSolver','termination':str(solution.termination),'acceleratorProbe':gpu,'mappingRequired':True},'cpu')
pap_export(job,result,job_is_demo,'pap-pybamm-result.json')